<a href="https://colab.research.google.com/github/DeepLabCut/DeepLabCut/blob/master/examples/COLAB/COLAB_YOURDATA_TrainNetwork_VideoAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

# DeepLabCut 用于您的标准（单动物）项目！

一些有用的链接：

- [DeepLabCut 的 GitHub: github.com/DeepLabCut/DeepLabCut](https://github.com/DeepLabCut/DeepLabCut)
- [DeepLabCut 的文档：单动物项目用户指南](https://deeplabcut.github.io/DeepLabCut/docs/standardDeepLabCut_UserGuide.html)


本 Notebook（指南）说明了如何使用云平台来完成以下操作：
- 创建训练数据集
- 训练神经网络
- 评估网络性能
- 创建简单的质量检查图表
- 分析新的视频！

### 本 Notebook 假设您已经拥有一个包含已标记数据的项目文件夹！

本 Notebook 将演示如何将 DeepLabCut 用于您自己项目的必要步骤。

这里展示了完成此操作的最简单代码，但许多函数都具有额外的功能，因此请务必查阅概述文档和该研究论文！

Nath\*, Mathis\* 等人：《Using DeepLabCut for markerless pose estimation during behavior across species. Nature Protocols, 2019.》（在跨物种行为研究中使用 DeepLabCut 进行无标记姿态估计）。


论文：https://www.nature.com/articles/s41596-019-0176-0

预印本：https://www.biorxiv.org/content/biorxiv/early/2018/11/24/476531.full.pdf

## 首先，进入 “Runtime” -> “change runtime type” -> 选择 “Python3”，然后选择 “GPU”

由于 COLAB 环境已更新到 CUDA 12.X 和 Python 3.11，我们需要以不同的方式安装 DeepLabCut 和 TensorFlow，以确保 TensorFlow 能够成功连接到 GPU。

In [ ]:
# this will take a couple of minutes to install all the dependencies!
!pip install --pre deeplabcut

（请确保在继续之前点击上方显示的 "RESTART RUNTIME"！）您会在上方单元格的输出中看到此按钮 ^。

In [2]:
import deeplabcut

DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)


## 关联您的 Google Drive（包含您的标注数据或演示数据）：

### 首先，请将您的项目文件夹放入 Google Drive 中！“例如：将名为 "Project-YourName-TheDate" 的文件夹移动到 Google Drive 中。

In [ ]:
# Now, let's link to your GoogleDrive. Run this cell and follow the authorization instructions:
# (We recommend putting a copy of the github repo in your google drive if you are using the demo "examples")

from google.colab import drive

drive.mount("/content/drive")

您需要编辑 **`config.yaml` 文件中的项目路径**，将其设置为您的 Google Drive 链接！

通常情况下，这个路径会是：`/content/drive/My Drive/yourProjectFolderName`

In [ ]:
# PLEASE EDIT THIS:
project_folder_name = "MontBlanc-Daniel-2019-12-16"
video_type = "mp4" #, mp4, MOV, or avi, whatever you uploaded!

# No need to edit this, we are going to assume you put videos you want to analyze
# in the "videos" folder, but if this is NOT true, edit below:
videofile_path = [f"/content/drive/My Drive/{project_folder_name}/videos/"]
print(videofile_path)

# The prediction files and labeled videos will be saved in this `labeled-videos` folder
# in your project folder; if you want them elsewhere, you can edit this;
# if you want the output files in the same folder as the videos, set this to an empty string.
destfolder = f"/content/drive/My Drive/{project_folder_name}/labeled-videos"

#No need to edit this, as you set it when you passed the ProjectFolderName (above):
path_config_file = f"/content/drive/My Drive/{project_folder_name}/config.yaml"
print(path_config_file)

# This creates a path variable that links to your Google Drive project

## 创建训练数据集：

### 您必须在 Colab 中执行此步骤

运行此脚本后，训练数据集将在项目目录下的 **'training-datasets'** 子目录中创建并保存。

此函数还会**在 `dlc-models-pytorch` 下创建新的子目录**，并用正确的训练和测试姿态配置文件的路径更新 `project config.yaml` 文件。这些文件保存了训练网络的参数。工具箱中提供了一个示例文件，名为 **`pytorch_config.yaml`**。

现在是时候开始训练网络了！

In [ ]:
# There are many more functions you can set here, including which network to use!
# Check the docstring for `create_training_dataset` for all options you can use!

deeplabcut.create_training_dataset(
  path_config_file, net_type="resnet_50", engine=deeplabcut.Engine.PYTORCH
)

## 开始训练：
此函数针对训练数据集的一个特定洗牌（shuffle）对网络进行训练。

In [ ]:
# Let's also change the display and save_epochs just in case Colab takes away
# the GPU... If that happens, you can reload from a saved point using the
# `snapshot_path` argument to `deeplabcut.train_network`:
#   deeplabcut.train_network(..., snapshot_path="/content/.../snapshot-050.pt")

# Typically, you want to train to ~200 epochs. We set the batch size to 8 to
# utilize the GPU's capabilities.

# More info and there are more things you can set:
#   https://deeplabcut.github.io/DeepLabCut/docs/standardDeepLabCut_UserGuide.html#g-train-the-network

deeplabcut.train_network(
    path_config_file,
    shuffle=1,
    save_epochs=5,
    epochs=200,
    batch_size=8,
)

# This will run until you stop it (CTRL+C), or hit "STOP" icon, or when it hits the end.

请注意，当您按下 "STOP" 时，您会收到一个 `KeyboardInterrupt` “错误”！不用担心！ :)

## 开始评估：
此函数用于评估某个特定洗牌（shuffle/shuffles）下、在特定状态或所有状态下的训练模型在数据集（图像）上的表现，并将结果作为 `.csv` 文件存储在 `evaluation-results-pytorch` 目录下的一个子目录中。

In [ ]:
deeplabcut.evaluate_network(path_config_file, plotting=True)

# Here you want to see a low pixel error! Of course, it can only be as
# good as the labeler, so be sure your labels are good!


## 您可以在 Colab 外部执行一个可选的优化步骤：

- 如果您的像素误差（pixel errors）没有降到足够低的水平，请查阅协议指南，了解如何精炼（refine）您的网络模型！
- 您将需要在 **Colab 环境之外**调整标签！我们建议您返回，继续训练和分析视频……
- 请参阅代码仓库（repo）和协议说明，了解如何优化您的数据！

## 开始分析视频：

此函数用于分析新上传的视频。用户可以从评估结果中选择最佳模型，并在 `config.yaml` 文件中为变量 `snapshotindex` 指定正确的快照索引。否则，系统将默认使用最新的快照来分析视频。

分析结果将以 hd5 文件的形式存储在与视频所在的同一个目录下。

In [ ]:
deeplabcut.analyze_videos(
    path_config_file,
    videofile_path,
    videotype=video_type,
    destfolder=destfolder,
)

## 绘制所分析视频的轨迹：
此函数会绘制整个视频中所有身体部位的轨迹。每个身体部位都由一个唯一的颜色进行标识。

In [ ]:
deeplabcut.plot_trajectories(
    path_config_file,
    videofile_path,
    videotype=video_type,
    destfolder=destfolder,
)

现在你可以查看 `plot-poses` 文件，并检查 `plot-likelihood.png` 文件，你可能需要在 `config.yaml` 文件中修改 `"p-cutoff"` 的值，以便在视频中只显示高置信度的点。例如，可以设置为大约 `0.8` 或 `0.9`。当前的默认值是 `0.4`。

## 创建带标签的视频：
此函数仅用于可视化目的，可用于根据网络预测的标签创建 .mp4 格式的视频。此视频将保存到原始视频所在的同一目录下。

In [ ]:
deeplabcut.create_labeled_video(
    path_config_file,
    videofile_path,
    videotype=video_type,
    destfolder=destfolder,
)